# Objetivo do projeto e contexto (prever preços de carros para a empresa Cars 4 You).

# Import Libraries

In [58]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# Create Meta Data

**carID:** An atribute that contains an identifier for each car.

**Brand:** The cars main brand (e.g., Ford, Toyota).

**model:** The car model.

**year:** The year of registration of the car.

**mileage:** The total reported distance travelled by the car (in miles).

**tax:** The amount of road tax (in £) that, in 2020, was applicable to the car in question.

**fuelType:** Type of Fuel used by car (Diesel, Petrol, Hybrid, Electric).

**mpg:** Average Miles per Gallon.

**engineSize:** Size of Engine in liters (Cubic Decimeters).

**paintQuality%** The mechanic’s assessment of the cars’ overall paint quality and hull integrity (filled by the mechanic during evaluation). 

**previousOwner:** Number of previous registered owners of the vehicle.

**hasDamage:** Boolean marker filled by the seller at the time of registration stating whether the car is damaged or not.

**price:** The car's price when purchased by Cars 4 You (in £).

# Import Dataset

In [59]:
import os
os.listdir('../data')

['test.csv', '.gitkeep', 'train.csv', 'sample_submission.csv']

In [60]:
sample = pd.read_csv('../data/sample_submission.csv')
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

# EDA (Exploratory Data Analysis)

## Define new index for our datasets

In [61]:
sample.set_index('carID', inplace = True)
train.set_index('carID', inplace = True)
test.set_index('carID', inplace = True)

## Sample

In [62]:
print("Initial Analysis of the Sample Submission Dataset")
print('')
print(sample.shape)
print('-----------------------')
print(sample.head())
print('-----------------------')
print(sample.tail())

Initial Analysis of the Sample Submission Dataset

(32567, 1)
-----------------------
         price
carID         
89856   851000
106581  514000
80886   323000
100174  921000
81376   620000
-----------------------
         price
carID         
105775  412000
81363   441000
76833   684000
91768   835000
99627     2000


In [63]:
print("Continue to analyse the Sample Submission Dataset")
print('------------------------')
print('')
print(sample.info()) #From this output we can see that this dataset hasn't missing values
print('')
print('------------------------')
print(sample.describe())

Continue to analyse the Sample Submission Dataset
------------------------

<class 'pandas.core.frame.DataFrame'>
Index: 32567 entries, 89856 to 99627
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   price   32567 non-null  int64
dtypes: int64(1)
memory usage: 508.9 KB
None

------------------------
               price
count   32567.000000
mean   498198.943716
std    288111.809635
min      1000.000000
25%    249000.000000
50%    501000.000000
75%    746000.000000
max    999000.000000


## Train

In [64]:
train.shape

(75973, 13)

In [65]:
train.head()

,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,,
69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.000000,0.0
53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.000000,0.0
6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.000000,0.0
29021,Ford,FIESTA,2018.0,12500,anual,9102.0,Petrol,145.0,65.700000,1.0,50.0,-2.340306,0.0
10062,BMW,2 Series,2019.0,22995,Manual,1000.0,Petrol,145.0,42.800000,1.5,97.0,3.000000,0.0


In [66]:
train.tail()

,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,,
37194,Mercedes,C Class,2015.0,13498,Manual,14480.0,etrol,125.0,53.3,2.0,78.0,0.0,0.0
6265,Audi,Q3,2013.0,12495,Semi-Auto,52134.0,Diesel,200.0,47.9,2.0,38.0,2.0,0.0
54886,Toyota,Aygo,2017.0,8399,Automatic,11304.0,Petrol,145.0,67.0,1.0,57.0,3.0,0.0
860,Audi,Q3,2015.0,12990,Manual,69072.0,iesel,125.0,60.1,2.0,74.0,2.0,0.0
15795,Ford,Fiesta,2018.0,10495,Manual,16709.0,Petro,145.0,64.2,1.1,38.0,1.0,0.0


From the visualization of the head and tail of the data base we can already understand that some errors exist:

    - Missing values
    - Values that should be integers as floats (2020.0)
We will further analyse this using describe and info.

It's also possible to see that some strings have the same information written in different forms (Diesel, iesel; Mercedes, mercedes).
To solve this problem we will uniformize all the values in data preparation

In [67]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 75973 entries, 69512 to 15795
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Brand           74452 non-null  object 
 1   model           74456 non-null  object 
 2   year            74482 non-null  float64
 3   price           75973 non-null  int64  
 4   transmission    74451 non-null  object 
 5   mileage         74510 non-null  float64
 6   fuelType        74462 non-null  object 
 7   tax             68069 non-null  float64
 8   mpg             68047 non-null  float64
 9   engineSize      74457 non-null  float64
 10  paintQuality%   74449 non-null  float64
 11  previousOwners  74423 non-null  float64
 12  hasDamage       74425 non-null  float64
dtypes: float64(8), int64(1), object(4)
memory usage: 8.1+ MB


From info we can see that:

    - year as a float...
    - previousOwners, hasDamage also as floats but they should be integers and booleans respectively
    - Missing values in all features except the price 

What will we do?

    Analyse with describe to have a different view

In [68]:
train.describe()

,year,price,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
count,74482.000000,75973.000000,74510.000000,68069.000000,68047.000000,74457.000000,74449.000000,74423.000000,74425.0
mean,2017.096611,16881.889553,23004.184088,120.329078,55.152666,1.660136,64.590667,1.994580,0.0
std,2.208704,9736.926322,22129.788366,65.521176,16.497837,0.573462,21.021065,1.472981,0.0
min,1970.000000,450.000000,-58540.574478,-91.121630,-43.421768,-0.103493,1.638913,-2.345650,0.0
25%,2016.000000,10200.000000,7423.250000,125.000000,46.300000,1.200000,47.000000,1.000000,0.0
50%,2017.000000,14699.000000,17300.000000,145.000000,54.300000,1.600000,65.000000,2.000000,0.0
75%,2019.000000,20950.000000,32427.500000,145.000000,62.800000,2.000000,82.000000,3.000000,0.0
max,2024.121759,159999.000000,323000.000000,580.000000,470.800000,6.600000,125.594308,6.258371,0.0


From the numeric describe we can see that we have some weird values:

    1. negative mileage, tax, mpg, engineSize, previousOwners in the minimum value
    2. hasDamage is a boolean but we can see that instead of 0 and 1 we only have 0 and Nones*
    3. previousOwner has a float? Should we round it?

*check in the hasDamage column

What will we do:

    1. Count the number of negative values and decide if we should drop or change them.
    2. Replace the nones by 1's. (data-preparation)
    3. Count the number of float values and decide to drop or round them.

In [69]:
train.describe(include='object')

,Brand,model,transmission,fuelType
count,74452,74456,74451,74462
unique,72,735,40,34
top,Ford,Focus,Manual,Petrol
freq,14808,6353,38050,37995


From the categorical describe we can confirm that this columns also have missing values


In [73]:
for column in train.columns:
    unique_values = train[column].unique()
    print(f"Unique values on the column '{column}': {unique_values}")
    print('-----------------------------------')
    print('')


Unique values on the column 'Brand': ['VW' 'Toyota' 'Audi' 'Ford' 'BMW' 'Skoda' 'Opel' 'Mercedes' 'FOR'
 'mercedes' 'Hyundai' 'w' 'ord' 'MW' 'bmw' nan 'yundai' 'BM' 'Toyot' 'udi'
 'Ope' 'AUDI' 'V' 'opel' 'pel' 'For' 'pe' 'Mercede' 'audi' 'MERCEDES'
 'OPEL' 'koda' 'FORD' 'Hyunda' 'W' 'Aud' 'vw' 'hyundai' 'skoda' 'ford'
 'TOYOTA' 'ercedes' 'oyota' 'toyota' 'SKODA' 'Skod' 'HYUNDAI' 'kod' 'v'
 'for' 'SKOD' 'aud' 'KODA' 'PEL' 'yunda' 'or' 'UDI' 'OYOTA' 'HYUNDA' 'mw'
 'OPE' 'mercede' 'ERCEDES' 'ercede' 'TOYOT' 'MERCEDE' 'ORD' 'ud' 'ope'
 'AUD' 'hyunda' 'skod' 'toyot']
-----------------------------------

Unique values on the column 'model': [' Golf' ' Yaris' ' Q2' ' FIESTA' ' 2 Series' '3 Series' ' A3' ' Octavia'
 ' Passat' ' Focus' ' Insignia' ' A Clas' ' Q3' ' Fabia' ' A Class' ' Ka+'
 ' 3 Series' ' GLC Class' ' I30' ' C Class' ' Polo' ' E Class' ' C Clas'
 ' Q5' ' Up' ' Fiesta' ' C-HR' ' Mokka X' ' Corsa' ' Astra' ' TT'
 ' 5 Series' ' Aygo' ' 4 Series' ' SLK' ' Viva' ' T-Roc' 'Focus'
 ' E

Comments on the results above for each feature:
- Brand have significant inconsistencies with case sensitivity, ('mercede', 'FOR'), and fragmented entries ('pel', 'udi').

- Model have a lot of variations due to leading/trailing spaces, inconsistent casing, typos ('Focu' instead of 'Focus'), and incomplete entries ('A Clas').

- Year have decimal/fractional values (e.g., 2023.36707842) suggests data encoding errors or floating-point artifacts. 

- Price have numeric and diverse.

- Transmission column with multiple variants and typos exist for common types (e.g., 'Manual', 'manua', 'anual'). 

- Mileage appears mostly clean and numeric.

- FuelType have many spelling and case variations (e.g., 'Petrol', 'petrol', 'etrol') exist.

- Tax with negative and fractional values are suspicious, suggesting data errors or unusual cases. 

- MPG looks like contain outliers and invalid negative values.

- EngineSize is mostly numeric but includes some unusual fractional and zero values that should be verified.

- PaintQuality% looks like have some outliers and very small unusual values are present that can make no sense.

- PreviousOwners contains unexpected negative/fractional values and NaNs.

- HasDamage is Binary flag with values 0 and NaN.

### Check for duplicates

In [ ]:
train.duplicated().sum()
#We conclude that there aren't any duplicates on the whole table

Besides not having duplicates it's important to examine again the duplicates without the carID, to prevent redundancy.

In [ ]:
#First we create a copy of our dataFrame 
train_copy= train.copy()
train_copy.head()

In [ ]:
#We drop carID
train_without_ID = train_copy.drop('carID', axis=1)
train_without_ID.head() #to check that everything is okay

In [ ]:
train_without_ID.duplicated().sum() 
#count the total of duplicates of our new DataFrame

After this analysis we found inly 4 duplicates, we decide to drop the duplicate lines. 

### GroupBy

In [ ]:
train.groupby('hasDamage')['price'].mean()

From the code before we can see that the price mean for the cars that have no damage is 16883.212509. 
It would also make sense to analyse the mean price of the cars with damage but the problem is that assume that we have cars with damage, because the column with the feature hasDamge has no 1's.
this way it's impossible to check the mean price of the cars with damage.

In [ ]:
train.groupby('mileage')['price'].mean()
#Here we can check a decrease of the price with the increase of the mileages
#But we still have the problem of the negatives mileages

In [ ]:
train.groupby('year')['price'].mean()
#It's possible to verify that the price varies depending ont he year of manufacture of the cars. 
#This is, the older the car, he cheaper it is

In [ ]:
!!!!! O CÓDIGO A SEGUIR FORAM RACIOCÍNEOS QUE FIZ QUE PODIAM DAR RESULTADOS GIROS MAS QUE NÃO SEI SE VALE A PENA POR  !!!!

train.groupby('previousOwners')['price'].mean()
#aqui podiamos relacionar o aumento do nº de donos com a diminuição do preço mas acho que os resultados não mostram nada muito
#interessante, não sei se vale a pena por

train.groupby('Brand')['price'].mean()
#pode ser interessante ver depois de ter corrigido o problema dos erros na escrita de cada marca. 
# é claro pelo resultado que por exemplo os yundais sao mais baratos que maior parte das outras marcas

train.groupby('paintQuality%')['price'].mean()
#acho que o resultado não mostra nada que tenha uma boa conclusao

In [ ]:
!!!!! O CÓDIGO A SEGUIR FORAM RACIOCÍNEOS QUE FIZ QUE PODIAM DAR RESULTADOS GIROS MAS QUE NÃO SEI SE VALE A PENA POR  !!!!

train.groupby('year')['paintQuality%'].mean()
#poderiamos ver que quanto mais velho o carro pior a qualidade de pintura, mas acho que isso não é muito explicito pelos resultados

train.groupby('previousOwners')['paintQuality%'].mean()
#os resultados tb nao permitem concluir nada

train.groupby('hasDamage')['paintQuality%'].mean()
#acho que aqui era giro comparar a qualidade de pintura com os carros que têm e não têm estragos mas como só temos missing values nos sitios dos 1's nao conseguimos concluir nada disso



### Define the independent variables as X and the dependent as Y

In [ ]:
X = train.drop('price', axis = 1)
y = train['price']

TO DO:
- corrigir types
- remover outliers graves, que são erros e visualizar boxplots
- Feature engineering: Criar features que não envolvem cálculos c/ média, mediana, …
Fazer tudo isto para o training and test set

### Split the dataset into train and validation

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X,y, test_size = 0.3, 
                                                  random_state = 0, 
                                                  #stratify = y- não pus esta parte como no notebook pq estava a dar erro e acho que é pq a nossa variavel y aqui não é um boolean mas sim um float
                                                  shuffle = True)

TO DO:

- Feature engineering: Criar features que envolvem cálculos c/ média, mediana, … (fazer para os 3 sets que temos)
- Preencher missing values nos 3 datasets, acho que dá para usar uma função ‘transform’ (justificar para depois por no report)  
!!!!!ATENÇÃO: os valores de média, moda,… a usar neste últimos dois passos são todos retirados apenas do training set e não do validation set

### Fill missing values

As we saw there are missing values in a couple of variables so we will fill the categories missing values with 'Unknow' and the numericals with the average.

But before do that we'll split our columns in metric and non_metric features.

In [ ]:
for column in ['Brand', 'model', 'transmission', 'fuelType']:
    X_train[column] = X_train[column].fillna('Unknown')
    X_val[column] = X_val[column].fillna('Unknown')

In [ ]:
for column in X_train.columns:
    if pd.api.types.is_numeric_dtype(X_train[column]):
        
        #store mean of training data in a variable - in a real application, you may need to store these values for future usages on e.g. test data 
        mean_to_fill = X_train[column].mean()
        
        #fill on X_train
        X_train[column].fillna(mean_to_fill, inplace=True)
        #Fill on X_val
        X_val[column].fillna(mean_to_fill, inplace=True)

## Test

In [ ]:
test.shape
#Here we can see that the test shape is equal to the sample shape.

(32567, 12)

In [ ]:
test.head(15)

,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,
89856,Hyundai,I30,2022.878006,Automatic,30700.000000,petrol,205.0,41.50000,1.6,61.0,3.0,0.0
106581,VW,Tiguan,2017.000000,Semi-Auto,-48190.655673,Petrol,150.0,38.20000,2.0,60.0,2.0,0.0
80886,BMW,2 Series,2016.000000,Automatic,36792.000000,Petrol,125.0,51.40000,1.5,94.0,2.0,0.0
100174,Opel,Grandland X,2019.000000,Manual,5533.000000,Petrol,145.0,44.10000,1.2,77.0,1.0,0.0
81376,BMW,1 Series,2019.000000,Semi-Auto,9058.000000,Diesel,150.0,51.40000,2.0,45.0,4.0,0.0
85391,Ford,Fiesta,2018.000000,Manual,29626.000000,Petrol,145.0,65.70000,1.0,64.0,1.0,0.0
82175,BMW,X1,2016.000000,Manual,57717.000000,Diese,125.0,58.90000,2.0,50.0,1.0,0.0
95250,Mercedes,B Class,2017.000000,Manual,14005.000000,Diesel,145.0,65.70000,NaN,64.0,4.0,0.0
85071,Ford,Focus,2011.000000,Manual,68274.000000,Petrol,145.0,47.90000,1.6,71.0,4.0,0.0


In [ ]:
test.tail(15)

,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,
90396,Hyundai,I10,2017.0,Manual,5970.00000,Petrol,150.0,60.1,1.0,75.0,0.0,0.0
97935,Toyota,Aygo,2019.0,Manual,10.00000,Petrol,145.0,56.5,1.0,89.0,2.0,0.0
80399,BMW,5 Series,2019.0,Automatic,1522.00000,Hybrid,135.0,141.3,2.0,38.0,3.0,0.0
105883,VW,Polo,2017.0,Manual,19812.00000,Petrol,145.0,60.1,1.2,71.0,1.0,0.0
92823,Mercedes,C Clas,2019.0,Semi-Auto,10545.00000,etrol,NaN,NaN,1.5,NaN,1.0,0.0
82238,BMW,i3,2017.0,AUTOMATIC,19178.00000,Other,0.0,470.8,0.6,43.0,0.0,0.0
98091,Toyota,Prius,2016.0,Automatic,28000.00000,hybrid,0.0,85.6,1.8,46.0,4.0,0.0
87257,Ford,Fiesta,2017.0,Semi-Auto,4512.00000,Petrol,0.0,57.7,1.0,35.0,0.0,0.0
87937,Ford,Fiesta,2018.0,Manual,10297.00000,Petrol,150.0,64.2,1.1,68.0,3.0,0.0


From the visualization of the head and tail of the data base we can already understand that some errors exist:

    - Missing values
    - Values in the columns Year, hasDamage, previousOwners that should be integers as floats (2020.0)
    - Floats on mpg, previousOwners column with diferent sizes
    - A category unknown in transmission column
    - It looks like the column hasDamage only contains 0's and blanks/None values, are the blanks supose to be 1's?
    - PreviousOwner and tax: negative values are impossible
We will further analyse this using describe and info.

It's also possible to see that some strings have the same information written in different forms (Petrol as etrol, Automatic and AUTOMATIC).
To solve this problem we will uniformize all the values in data preparation

In [ ]:
test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32567 entries, 89856 to 99627
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Brand           31918 non-null  object 
 1   model           31917 non-null  object 
 2   year            31914 non-null  float64
 3   transmission    31944 non-null  object 
 4   mileage         31878 non-null  float64
 5   fuelType        31911 non-null  object 
 6   tax             29259 non-null  float64
 7   mpg             29279 non-null  float64
 8   engineSize      31939 non-null  float64
 9   paintQuality%   31942 non-null  float64
 10  previousOwners  31970 non-null  float64
 11  hasDamage       31970 non-null  float64
dtypes: float64(8), object(4)
memory usage: 3.2+ MB


From info we can see that:

    - year, as a float...
    - previousOwners, hasDamage also as floats but they should be integers and booleans respectively
    - Missing values in all features

What will we do?

    Analyse with describe to have a different view

In [ ]:
test.describe()

,year,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
count,31914.000000,31878.000000,29259.000000,29279.000000,31939.000000,31942.000000,31970.000000,31970.0
mean,2017.102299,22952.658921,120.569239,55.210728,1.665377,64.446667,2.006118,0.0
std,2.207969,22132.758713,65.560570,17.644635,0.574467,21.142188,1.472310,0.0
min,1991.000000,-58540.574478,-91.121630,-43.421768,-0.103493,1.638913,-2.345650,0.0
25%,2016.000000,7298.250000,125.000000,46.300000,1.200000,47.000000,1.000000,0.0
50%,2017.000000,17225.500000,145.000000,54.300000,1.600000,65.000000,2.000000,0.0
75%,2019.000000,32500.000000,145.000000,62.800000,2.000000,82.000000,3.000000,0.0
max,2024.121759,279000.000000,580.000000,470.800000,6.600000,125.594308,6.258371,0.0


From the numeric describe we can see that we have some weird values:

    1. negative mileage, tax, mpg, engineSize, previousOwners in the minimum value
    2. hasDamage is a boolean but we can see that instead of 0 and 1 we only have 0 and Nones*
    3. previousOwner has a float? Should we round it?

*check in the hasDamage column

What will we do:

    1. Count the number of negative values and decide if we should drop or change them.
    2. Replace the nones by 1's. (data-preparation)
    3. Count the number of float values and decide to drop or round them.

In [ ]:
test.describe(include='object')

,Brand,model,transmission,fuelType
count,31918,31917,31944,31911
unique,64,593,38,29
top,Ford,Focus,Manual,Petrol
freq,6360,2721,16312,16113


From the categorical describe we tell that:

    - This columns also have missing values
    - transmission has 38 unique values and fuelType has 29...


In [ ]:
train.unique()

AttributeError: 'DataFrame' object has no attribute 'unique'

# Data Preprocessing

What should we do with the negative values? First of all we'll analyse them.

In [ ]:
mileage_train_negatives = train['mileage']<0
tax_train_negatives = train['tax']<0
mpg_train_negatives = train['mpg']<0
engineSize_train_negatives = train['engineSize']<0
previousOwners_train_negatives = train['previousOwners']<0


#here we used chatGPT to help us to construct the following DataFrame
negatives_summary = pd.DataFrame({
    'mileage_negatives': mileage_train_negatives.value_counts(),
    'tax_negatives': tax_train_negatives.value_counts(),
    'mpg_negatives': mpg_train_negatives.value_counts(),
    'engineSize_negatives': engineSize_train_negatives.value_counts(),
    'previousOwners_negatives': previousOwners_train_negatives.value_counts()
})

negatives_summary

In [ ]:
#Check the percentage of negative mileage values
data_len = len(train['mileage'])

#Here we used chatGPT to help us constructing the final negatives_table
negatives_percent = {
    'mileage_negatives (%)': mileage_train_negatives.sum() / data_len * 100,
    'tax_negatives (%)': tax_train_negatives.sum() / data_len * 100,
    'mpg_negatives (%)': mpg_train_negatives.sum() / data_len * 100,
    'engineSize_negatives (%)': engineSize_train_negatives.sum() / data_len * 100,
    'previousOwners_negatives (%)': previousOwners_train_negatives.sum() / data_len * 100
}

# Converter para DataFrame (tabela)
negatives_table = pd.DataFrame(negatives_percent, index=['Percentage of Negatives']).round(3)

negatives_table

From the negatives_table we obtain very low percentages for each variable, lower than 1%. So it's possible to conclude that this values are a small part of our data and that if we drop them we don't loose veracity.

We can also see that the total percentage of negative values is lower then 5%.

In [ ]:
# The goal is to count the values that are floats, this is, the values that have decimals different than 0
# To do this we create a variable with only the values that have decimals cases different than 0, that is, 
#the values for which the remainder of the division by 1 is not 0
has_decimals = train['previousOwners'] % 1 != 0
print(has_decimals.sum())
#at the end we sum all those values to find how many "real" floats exist


has_decimals.sum() / data_len * 100 

In [ ]:
brand_same_size = train['Brand'].str.lower()
brand_same_size.value_counts().to_frame()

In [ ]:
mapping_brands = {
    'ord' : 'ford',
    'for' : 'ford',
    'ercedes' : 'mercedes',
    'mercede' : 'mercedes',
    'w' : 'vw',
    'v' : 'vw',
    'ope' : 'opel',
    'pel' : 'opel',
    'mw' : 'bmw',
    'aud' : 'audi',
    'udi' : 'audi',
    'bm' : 'bmw',
    'oyota' : 'toyota',
    'koda' : 'skoda',
    'skod' : 'skoda',
    'toyot' : 'toyota',
    'yundai' : 'hyundai',
    'hyunda' : 'hyundai',
    'ercede' : 'mercedes',
    'or' : 'ford',
    'pe' : 'opel',
    'yunda' : 'hyundai',
    'ud' : 'audi',
    'kod' : 'skoda'
}
#train['Brand'] = train['Brand].replace(mapping_brands)

podemos usar esta maneira ou uma biblioteca chamada fuzzy matching mas não sei se vai ser permitido

hasDamage column - replace None values by 1

In [ ]:
train['hasDamage']=train['hasDamage'].fillna(1) #do the same for test set?

In [ ]:
#validate the changes, isto podemos apagar depois
train['hasDamage'].value_counts() 

previousOwner column - round to transform float into int

In [ ]:
train['previousOwners'] = train['previousOwners'].round(0) #do the same for test set?

In [ ]:
train['previousOwners'].value_counts() #negative values remove or stay? isto podemos apagar depois